<div style="background:#12314a;color:#ffffff;padding:26px 30px;border-radius:10px;border-left:7px solid #22a6a1;">
  <div style="font-size:12px;letter-spacing:2px;color:#9ee3df;">MURSHID | TECHNICAL ASSIGNMENTS</div>
  <h1 style="color:#ffffff;margin:12px 0 8px 0;font-size:30px;">Day 01 — Local Model Demonstration</h1>
  <p style="color:#d9e8f2;font-size:15px;margin:0">Environment validation, model loading, and grounded text generation</p>
</div>

**Author:** Setah Saad  
**Training program:** LLM Application Engineering · SDA-AIE-213  
**Institution:** [SDAIA Academy](https://github.com/SDAIAAcademy)  
**Notebook type:** Independent practical homework  
**Documentation language:** English

---

<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">01 | Assignment overview</h2></div>

**Purpose.** Set up a local CPU-based demonstration with a small pretrained sequence-to-sequence model and a public, synthetic service question.

**Learning objectives**

- Confirm the notebook environment and inspect installed library versions.
- Configure a specific model and revision for a reproducible model selection.
- Run deterministic generation on synthetic context and capture timing and token counts.
- Save the generated response and metadata as a JSON artifact.

**Technical environment**

| Component | Usage |
|:--|:--|
| Runtime | Python 3 · Google Colab |
| Libraries | PyTorch, Transformers, SentencePiece |
| Model | google/flan-t5-small (pretrained; not fine-tuned in this notebook) |
| Compute | CPU |
| Output | murshid_live_outputs/live_demo_result.json |

**Execution flow**

`Install and inspect → Configure model → Load tokenizer and model → Generate → Record metrics → Export JSON`

<div style="border-left:4px solid #087f83;background:#eef7f7;padding:12px 16px;margin:12px 0"><b>Scope and limitations</b><br>This is an optional real-model demonstration, separate from the core mock-client project. The notebook defines `run_live_demo()` but does **not** invoke it; the model will not generate an answer unless the function is called. Model files must be downloaded. No external paid inference API is used.</div>


<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">02 | Environment setup</h2></div>

Install the required libraries, import them, display their versions, and check CUDA availability. These checks help diagnose dependency and runtime differences before model execution.

**Expected observation:** Python and package versions, selected device, and installation status. The versions are environment-dependent; `pip install -U` does not pin them.


In [1]:

# Install required libraries
%pip install -U transformers sentencepiece

# Import libraries
import sys
import torch
import transformers
import sentencepiece

# Check Python version
print("Python version:", sys.version)

# Check library versions
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("SentencePiece version:", sentencepiece.__version__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Installation status
print("All libraries installed successfully!")
print("Environment is ready!")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 35.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.5 MB/s  0:00:00
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.2
    Uninstalling sentencepiece-0.2.2:
      Successfully uninstalled sentencepiece-0.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Torch version: 2.14.1+cpu
Transformers version: 5.23.0
SentencePiece version: 0.2.3
Device: cpu
All libraries installed successfully!
Environment is ready!


<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">03 | Model configuration</h2></div>

Define the model identifier, a fixed Hugging Face revision, and the local directory for generated artifacts. Pinning the **model revision** helps identify the model snapshot, while the environment setup above still uses unpinned package versions.

**Expected output:** No console output; constants and a `Path` object are created.


In [2]:
# OPTIONAL REAL-MODEL EXTENSION: run in a separate notebook/runtime.
# Never paste into or replace cells in the required mock notebook.
import json
import time
from pathlib import Path

LIVE_MODEL_ID = 'google/flan-t5-small'
LIVE_MODEL_REVISION = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'
LIVE_OUTPUT_DIR = Path('murshid_live_outputs')






<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">04 | Model inference and artifact generation</h2></div>

`run_live_demo()` loads the tokenizer and pretrained model on CPU, creates a prompt from a **public synthetic context** and a service question, then performs deterministic token generation using `do_sample=False`.

| Input | Meaning |
|:--|:--|
| `context` | Public service guidance; not a private customer record |
| `question` | Sample application-status question |
| `LIVE_MODEL_ID` / `LIVE_MODEL_REVISION` | Identifies the selected model snapshot |

**Function behavior:** Generates one answer, checks that the output is a nonempty string, measures inference duration, counts tokens, and writes a JSON record. A nonempty answer is only a **smoke check**, not proof of correctness, grounding, or safety.

**Artifact:** `murshid_live_outputs/live_demo_result.json` is written **only when the function runs**.


In [3]:
def run_live_demo():
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    torch.set_num_threads(2)
    tokenizer = AutoTokenizer.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION, trust_remote_code=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION,
        trust_remote_code=False, use_safetensors=True).to('cpu').eval()
    # Public synthetic context only; no private application records or identities.
    context = 'You can check your application status using its reference number.'
    question = 'How can I check my application status?'
    prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**encoded, max_new_tokens=48, do_sample=False, num_beams=1)
    answer = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
    # Smoke check only: this does NOT establish factual correctness or safety.
    assert isinstance(answer, str) and answer, 'No answer generated; inspect the runtime.'
    report = {
        'mode': 'real_local_model', 'model': LIVE_MODEL_ID,
        'model_revision': LIVE_MODEL_REVISION, 'device': 'cpu',
        'context': context, 'question': question, 'answer': answer,
        'inference_seconds': round(time.perf_counter() - started, 3),
        'input_tokens': int(encoded['input_ids'].shape[-1]),
        'output_tokens_including_special': int(generated.shape[-1]),
        'external_inference_api_calls': 0,
        'review_required': 'Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims.'
    }
    LIVE_OUTPUT_DIR.mkdir(exist_ok=True)
    (LIVE_OUTPUT_DIR / 'live_demo_result.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report




<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">90 | Technical review and references</h2></div>

### Engineering takeaways
- Demonstrates pretrained sequence-to-sequence inference without a hosted inference endpoint.
- Distinguishes a basic smoke test from semantic or security evaluation.
- Captures model metadata and latency for traceability.

### Validation boundaries
Generated text must be reviewed for factual support; this exercise does not establish production-grade safety. A function definition alone does not produce a saved result.

### Further reading
- [Transformers documentation](https://huggingface.co/docs/transformers/index)
- [Model card: FLAN-T5 small](https://huggingface.co/google/flan-t5-small)
- [PyTorch inference mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)
- [Python JSON documentation](https://docs.python.org/3/library/json.html)

---
*This notebook documents an educational implementation. Any reported results should be interpreted using the actual executed outputs and the limitations stated above.*
